# import libraries

In [40]:
# ==============================
# CELL 1: IMPORT LIBRARIES
# ==============================

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Load datasets

In [41]:
train_df = pd.read_csv("datacsv/train.csv")
train_df

,Num,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,1,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Moving Backward,Slight Injury
1,2,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
2,3,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,...,Going straight,Driver or rider,Male,31-50,3,Driver,NaN,Not a Pedestrian,Changing lane to the left,Serious Injury
3,4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,...,Going straight,Pedestrian,Female,18-30,3,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
4,5,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8205,8206,17:40:00,Monday,18-30,Male,Junior high school,Employee,5-10yr,Lorry (41?100Q),Owner,...,Going straight,Passenger,Male,31-50,3,Self-employed,Normal,Not a Pedestrian,Driving carelessly,Slight Injury
8206,8207,17:45:00,Tuesday,Over 51,Male,High school,Employee,5-10yr,Pick up upto 10Q,Owner,...,Unknown,Passenger,Female,Over 51,3,Self-employed,Normal,Not a Pedestrian,Changing lane to the left,Slight Injury
8207,8208,17:45:00,Tuesday,18-30,Male,Junior high school,Employee,5-10yr,Other,Owner,...,Turnover,Passenger,Female,31-50,3,Driver,Normal,Not a Pedestrian,Moving Backward,Slight Injury
8208,8209,8:25:00,Thursday,18-30,Male,Junior high school,Employee,Above 10yr,Lorry (41?100Q),Owner,...,Getting off,Passenger,Female,31-50,3,Employee,Normal,Not a Pedestrian,Overtaking,Slight Injury


In [4]:
train_df.shape

(8210, 33)

In [42]:
test_df = pd.read_csv("datacsv/test.csv")
test_df

,Num,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,...,Number_of_casualties,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident
0,8211,9:02:00,Monday,18-30,Male,Junior high school,Employee,2-5yr,Automobile,NaN,...,1,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Changing lane to the right
1,8212,9:02:00,Monday,31-50,Male,Junior high school,Employee,1-2yr,Automobile,Owner,...,1,Going straight,Driver or rider,Male,18-30,3,NaN,NaN,Not a Pedestrian,Overtaking
2,8213,20:10:00,Sunday,Over 51,Male,NaN,NaN,NaN,Stationwagen,Owner,...,2,Turnover,Driver or rider,Male,Over 51,2,NaN,NaN,Not a Pedestrian,Overloading
3,8214,20:10:00,Sunday,Over 51,Male,Elementary school,Employee,5-10yr,Other,Owner,...,2,Reversing,Passenger,Male,5,3,Self-employed,Normal,Not a Pedestrian,Other
4,8215,5:55:00,Thursday,Over 51,Male,Junior high school,Employee,5-10yr,Lorry (41?100Q),Governmental,...,2,Moving Backward,Pedestrian,Male,18-30,3,Driver,Normal,Unknown or other,No priority to vehicle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4101,12312,16:15:00,Wednesday,31-50,Male,NaN,Employee,2-5yr,Lorry (11?40Q),Owner,...,1,Going straight,na,na,na,na,Driver,Normal,Not a Pedestrian,No distancing
4102,12313,18:00:00,Sunday,Unknown,Male,Elementary school,Employee,5-10yr,Automobile,Owner,...,1,Other,na,na,na,na,Driver,Normal,Not a Pedestrian,No distancing
4103,12314,13:55:00,Sunday,Over 51,Male,Junior high school,Employee,5-10yr,Bajaj,Owner,...,1,Other,Driver or rider,Male,31-50,3,Driver,Normal,Not a Pedestrian,Changing lane to the right
4104,12315,13:55:00,Sunday,18-30,Female,Junior high school,Employee,Above 10yr,Lorry (41?100Q),Owner,...,1,Other,na,na,na,na,Driver,Normal,Not a Pedestrian,Driving under the influence of drugs


In [6]:
test_df.shape

(4106, 32)

In [7]:
print(train_df.columns)

Index(['Num', 'Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver',
       'Educational_level', 'Vehicle_driver_relation', 'Driving_experience',
       'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle',
       'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians',
       'Road_allignment', 'Types_of_Junction', 'Road_surface_type',
       'Road_surface_conditions', 'Light_conditions', 'Weather_conditions',
       'Type_of_collision', 'Number_of_vehicles_involved',
       'Number_of_casualties', 'Vehicle_movement', 'Casualty_class',
       'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity',
       'Work_of_casuality', 'Fitness_of_casuality', 'Pedestrian_movement',
       'Cause_of_accident', 'Accident_severity'],
      dtype='object')


In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8210 entries, 0 to 8209
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Num                          8210 non-null   int64 
 1   Time                         8210 non-null   object
 2   Day_of_week                  8210 non-null   object
 3   Age_band_of_driver           8210 non-null   object
 4   Sex_of_driver                8210 non-null   object
 5   Educational_level            7734 non-null   object
 6   Vehicle_driver_relation      7870 non-null   object
 7   Driving_experience           7665 non-null   object
 8   Type_of_vehicle              7558 non-null   object
 9   Owner_of_vehicle             7889 non-null   object
 10  Service_year_of_vehicle      5505 non-null   object
 11  Defect_of_vehicle            5225 non-null   object
 12  Area_accident_occured        8050 non-null   object
 13  Lanes_or_Medians             7943

In [9]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4106 entries, 0 to 4105
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Num                          4106 non-null   int64 
 1   Time                         4106 non-null   object
 2   Day_of_week                  4106 non-null   object
 3   Age_band_of_driver           4106 non-null   object
 4   Sex_of_driver                4106 non-null   object
 5   Educational_level            3841 non-null   object
 6   Vehicle_driver_relation      3867 non-null   object
 7   Driving_experience           3822 non-null   object
 8   Type_of_vehicle              3808 non-null   object
 9   Owner_of_vehicle             3945 non-null   object
 10  Service_year_of_vehicle      2883 non-null   object
 11  Defect_of_vehicle            2664 non-null   object
 12  Area_accident_occured        4027 non-null   object
 13  Lanes_or_Medians             3988

In [10]:
train_df.dtypes

Num                             int64
Time                           object
Day_of_week                    object
Age_band_of_driver             object
Sex_of_driver                  object
Educational_level              object
Vehicle_driver_relation        object
Driving_experience             object
Type_of_vehicle                object
Owner_of_vehicle               object
Service_year_of_vehicle        object
Defect_of_vehicle              object
Area_accident_occured          object
Lanes_or_Medians               object
Road_allignment                object
Types_of_Junction              object
Road_surface_type              object
Road_surface_conditions        object
Light_conditions               object
Weather_conditions             object
Type_of_collision              object
Number_of_vehicles_involved     int64
Number_of_casualties            int64
Vehicle_movement               object
Casualty_class                 object
Sex_of_casualty                object
Age_band_of_

In [11]:
test_df.dtypes

Num                             int64
Time                           object
Day_of_week                    object
Age_band_of_driver             object
Sex_of_driver                  object
Educational_level              object
Vehicle_driver_relation        object
Driving_experience             object
Type_of_vehicle                object
Owner_of_vehicle               object
Service_year_of_vehicle        object
Defect_of_vehicle              object
Area_accident_occured          object
Lanes_or_Medians               object
Road_allignment                object
Types_of_Junction              object
Road_surface_type              object
Road_surface_conditions        object
Light_conditions               object
Weather_conditions             object
Type_of_collision              object
Number_of_vehicles_involved     int64
Number_of_casualties            int64
Vehicle_movement               object
Casualty_class                 object
Sex_of_casualty                object
Age_band_of_

In [12]:
train_df.isnull().sum()

Num                               0
Time                              0
Day_of_week                       0
Age_band_of_driver                0
Sex_of_driver                     0
Educational_level               476
Vehicle_driver_relation         340
Driving_experience              545
Type_of_vehicle                 652
Owner_of_vehicle                321
Service_year_of_vehicle        2705
Defect_of_vehicle              2985
Area_accident_occured           160
Lanes_or_Medians                267
Road_allignment                 102
Types_of_Junction                 0
Road_surface_type               115
Road_surface_conditions           0
Light_conditions                  0
Weather_conditions                0
Type_of_collision               100
Number_of_vehicles_involved       0
Number_of_casualties              0
Vehicle_movement                184
Casualty_class                    0
Sex_of_casualty                   0
Age_band_of_casualty              0
Casualty_severity           

In [13]:
test_df.isnull().sum()

Num                               0
Time                              0
Day_of_week                       0
Age_band_of_driver                0
Sex_of_driver                     0
Educational_level               265
Vehicle_driver_relation         239
Driving_experience              284
Type_of_vehicle                 298
Owner_of_vehicle                161
Service_year_of_vehicle        1223
Defect_of_vehicle              1442
Area_accident_occured            79
Lanes_or_Medians                118
Road_allignment                  40
Types_of_Junction               887
Road_surface_type                57
Road_surface_conditions           0
Light_conditions                  0
Weather_conditions                0
Type_of_collision                55
Number_of_vehicles_involved       0
Number_of_casualties              0
Vehicle_movement                124
Casualty_class                    0
Sex_of_casualty                   0
Age_band_of_casualty              0
Casualty_severity           

# Check Duplicate Rows

In [14]:
train_df.duplicated().sum()

np.int64(0)

In [15]:
test_df.duplicated().sum()

np.int64(0)

# Remove ID Column

In [43]:
if "Num" in train_df.columns:
    train_df = train_df.drop(columns=["Num"])

if "Num" in test_df.columns:
    test_df = test_df.drop(columns=["Num"])

In [17]:
train_df.shape

(8210, 32)

In [18]:
test_df.shape

(4106, 31)

# Separate Features and Target

In [44]:
# ==============================
# CELL 5: SEPARATE X AND y
# ==============================

target = "Accident_severity"

X = train_df.drop(columns=[target])
y = train_df[target]

print("Number of input features:", X.shape[1])
print("Target column:", target)

print("\nTarget values:")
print(y.value_counts())

Number of input features: 31
Target column: Accident_severity

Target values:
Accident_severity
Slight Injury     7082
Serious Injury    1046
Fatal injury        82
Name: count, dtype: int64


In [45]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8210, 31)
y shape: (8210,)


# Train/Validation Split

In [46]:
# ==============================
# CELL 6: TRAIN VALIDATION SPLIT
# ==============================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (6568, 31)
X_val: (1642, 31)
y_train: (6568,)
y_val: (1642,)


# Identify Numerical and Categorical Features

In [47]:
# ==============================
# CELL 7: IDENTIFY FEATURE TYPES
# ==============================

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Numerical features:
['Number_of_vehicles_involved', 'Number_of_casualties']

Categorical features:
['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver', 'Educational_level', 'Vehicle_driver_relation', 'Driving_experience', 'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle', 'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians', 'Road_allignment', 'Types_of_Junction', 'Road_surface_type', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision', 'Vehicle_movement', 'Casualty_class', 'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity', 'Work_of_casuality', 'Fitness_of_casuality', 'Pedestrian_movement', 'Cause_of_accident']

Number of numerical features: 2
Number of categorical features: 29


# Create Preprocessing Pipeline

In [48]:
# ==============================
# CELL 8: PREPROCESSING
# ==============================

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


# Select Exactly 15 Features

In [49]:
# ==============================
# CELL 9: SELECT TOP 15 FEATURES
# ==============================

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=15
)

print("SelectKBest configured for 15 features.")

SelectKBest configured for 15 features.


# Select 15 Original Input Columns

In [50]:
# ==========================================
# CELL 10: SELECT EXACTLY 15 INPUT FEATURES
# ==========================================

selected_input_features = [
    "Time",
    "Day_of_week",
    "Age_band_of_driver",
    "Sex_of_driver",
    "Driving_experience",
    "Type_of_vehicle",
    "Area_accident_occured",
    "Types_of_Junction",
    "Road_surface_type",
    "Road_surface_conditions",
    "Light_conditions",
    "Weather_conditions",
    "Type_of_collision",
    "Number_of_vehicles_involved",
    "Number_of_casualties"
]

print("Selected 15 features:\n")

for i, feature in enumerate(selected_input_features, start=1):
    print(f"{i}. {feature}")

print("\nTotal features:", len(selected_input_features))

Selected 15 features:

1. Time
2. Day_of_week
3. Age_band_of_driver
4. Sex_of_driver
5. Driving_experience
6. Type_of_vehicle
7. Area_accident_occured
8. Types_of_Junction
9. Road_surface_type
10. Road_surface_conditions
11. Light_conditions
12. Weather_conditions
13. Type_of_collision
14. Number_of_vehicles_involved
15. Number_of_casualties

Total features: 15


In [51]:
# ==========================================
# CELL 11: VERIFY SELECTED FEATURES
# ==========================================

missing_features = [
    feature
    for feature in selected_input_features
    if feature not in train_df.columns
]

if missing_features:
    print("Missing features:")
    print(missing_features)
else:
    print("All 15 features are available in the dataset.")

All 15 features are available in the dataset.


# Create X Using Only 15 Features

In [52]:
# ==========================================
# CELL 12: CREATE X WITH 15 FEATURES
# ==========================================

X = train_df[selected_input_features].copy()

y = train_df["Accident_severity"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

X shape: (8210, 15)
y shape: (8210,)

X columns:
['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver', 'Driving_experience', 'Type_of_vehicle', 'Area_accident_occured', 'Types_of_Junction', 'Road_surface_type', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision', 'Number_of_vehicles_involved', 'Number_of_casualties']


# Train/Validation Split Again

In [53]:
# ==========================================
# CELL 13: TRAIN VALIDATION SPLIT
# ==========================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_train shape: (6568, 15)
X_val shape: (1642, 15)
y_train shape: (6568,)
y_val shape: (1642,)


In [54]:
# ==========================================
# CELL 14: IDENTIFY FEATURE TYPES
# ==========================================

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal numerical features:", len(numerical_features))
print("Total categorical features:", len(categorical_features))

Numerical features:
['Number_of_vehicles_involved', 'Number_of_casualties']

Categorical features:
['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver', 'Driving_experience', 'Type_of_vehicle', 'Area_accident_occured', 'Types_of_Junction', 'Road_surface_type', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision']

Total numerical features: 2
Total categorical features: 13


# Create Preprocessor for 15 Features

In [55]:
# ==========================================
# CELL 15: CREATE NEW PREPROCESSOR
# ==========================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("New preprocessor created successfully.")

New preprocessor created successfully.


# Create Random Forest Model

In [56]:
# ==============================
# CELL 15: RANDOM FOREST
# ==============================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

print("Random Forest model created.")

Random Forest model created.


# Create Complete Pipeline

In [57]:
# ==========================================
# CELL 17: CREATE COMPLETE PIPELINE
# ==========================================

pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            rf_model
        )
    ]
)

print("Complete pipeline created successfully.")

Complete pipeline created successfully.


# Train Model

In [58]:
# ==========================================
# CELL 18: TRAIN MODEL
# ==========================================

pipeline.fit(X_train, y_train)

print("Model training completed successfully.")

Model training completed successfully.


# Predict

In [59]:
# ==========================================
# CELL 19: MAKE VALIDATION PREDICTIONS
# ==========================================

y_pred = pipeline.predict(X_val)

print("Predictions generated successfully.")

print("\nFirst 10 predictions:")
print(y_pred[:10])

Predictions generated successfully.

First 10 predictions:
['Slight Injury' 'Slight Injury' 'Slight Injury' 'Slight Injury'
 'Slight Injury' 'Slight Injury' 'Slight Injury' 'Slight Injury'
 'Slight Injury' 'Slight Injury']


In [60]:
# ==========================================
# CELL 20: MODEL EVALUATION
# ==========================================

accuracy = accuracy_score(
    y_val,
    y_pred
)

precision = precision_score(
    y_val,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_val,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_val,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.8654080389768575
Precision: 0.8835783648083263
Recall   : 0.8654080389768575
F1 Score : 0.8057203990832397


In [61]:
# ==========================================
# CELL 21: CLASSIFICATION REPORT
# ==========================================

print(
    classification_report(
        y_val,
        y_pred,
        zero_division=0
    )
)

                precision    recall  f1-score   support

  Fatal injury       1.00      0.12      0.21        17
Serious Injury       1.00      0.01      0.03       209
 Slight Injury       0.86      1.00      0.93      1416

      accuracy                           0.87      1642
     macro avg       0.95      0.38      0.39      1642
  weighted avg       0.88      0.87      0.81      1642



In [62]:
# ==========================================
# CELL 22: CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(
    y_val,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[   2    0   15]
 [   0    3  206]
 [   0    0 1416]]


In [63]:
# ==========================================
# CELL 23: SAVE SELECTED FEATURES
# ==========================================

selected_features_df = pd.DataFrame({
    "Feature": selected_input_features
})

selected_features_df.to_csv(
    "selected_15_features.csv",
    index=False
)

print("selected_15_features.csv saved successfully.")

selected_15_features.csv saved successfully.


In [64]:
# ==========================================
# CELL 24: SAVE MODEL
# ==========================================

joblib.dump(
    pipeline,
    "accident_severity_model_15_features.pkl"
)

print(
    "Model saved as "
    "accident_severity_model_15_features.pkl"
)

Model saved as accident_severity_model_15_features.pkl
